In [1]:
%config IPCompleter.use_jedi = False
# %xmode Verbose
# %xmode context
%pdb off
%load_ext autoreload
%autoreload 3
# # Add exclusions for metaclass-using modules
# %aimport -neuropy.core.session.dataSession
# %aimport -neuropy.core.session.Formats.BaseDataSessionFormats
# %aimport -neuropy.core.session.Formats.Specific.KDibaOldDataSessionFormat
# %aimport -neuropy.core.session.Formats.Specific.BapunDataSessionFormat 
# %aimport -neuropy.core.session.Formats.Specific.RachelDataSessionFormat
# %aimport -neuropy.core.session.Formats.Specific.HiroDataSessionFormat

# !pip install viztracer
# %load_ext viztracer
# from viztracer import VizTracer

# %load_ext memory_profiler

import sys
from pathlib import Path
import numpy as np
import pyvista as pv
import time
import sys
import random
import queue
import os
import pythreejs
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from .example_animated_3D_sensor_quality import EEGVisualizer

os.environ['QT_API'] = 'pyqt5'
os.environ['PYQTGRAPH_QT_LIB'] = 'PyQt5'

# from PyQt5.QtWebEngineWidgets import QWebEngineView ## this must come first, before any QtApplication is made: 'ImportError: QtWebEngineWidgets must be imported or Qt.AA_ShareOpenGLContexts must be set before a QCoreApplication instance is created'

# required to enable non-blocking interaction:
%gui qt5

# In your __init__ method
pv.set_jupyter_backend('trame')

Automatic pdb calling has been turned OFF


ImportError: attempted relative import with no known parent package

In [ ]:
import numpy as np
import pyvista as pv
import time
import sys
import queue
import random
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# Electrode names (adjust based on your headset)
electrode_names = ['AF3', 'AF4', 'AuxCMS', 'AuxDRL', 'CMS', 'DRL', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8']

class EEGVisualizer:
    def __init__(self, model_path=None, update_interval=0.5):
        """
        Initialize the EEG visualizer
        
        Parameters:
        -----------
        model_path : str or Path, optional
            Path to the 3D model file (.glb or .obj)
        update_interval : float, optional
            Time interval between updates in seconds (default: 0.5)
        """
        self.update_interval = update_interval
        self.running = False
        
        # Set up PyVista visualization for Jupyter
        pv.set_jupyter_backend('trame')  # Use static backend for compatibility
        self.plotter = pv.Plotter(notebook=True)
        
        # Load the 3D model
        if model_path is None:
            model_path = Path('EXTERNAL/meshes/CompleteEmotivEpocEEG.glb').resolve()
        else:
            model_path = Path(model_path).resolve()
            
        if not model_path.exists():
            raise FileNotFoundError(f"Model file not found: {model_path}")
            
        print(f"Loading 3D model from: {model_path}")
        self.headset = pv.read(model_path)
        
        # Create electrode spheres at predefined positions
        # These are approximate positions for the Emotiv EPOC headset
        # You'll need to adjust these based on your actual model
        self.electrode_names_dict = {
            0:"AF3",
            1:"AF4",
            2:"Arm_R_Body",
            3:"ArmL_Body",
            4:"AuxCMS",
            5:"AuxDRL",
            6:"CMS",
            7:"DRL",
            8:"F3",
            9:"F4",
            10:"F7",
            11:"F8",
            12:"FC5",
            13:"FC6",
            14:"Headset_Back_Body",
            15:"O1",
            16:"O2",
            17:"P7",
            18:"P8",
            19:"T7",
            20:"T8",	
        }

        self.electrode_name_to_position_center = {
            'AF3': [-0.02849366665483203, -0.0731334302412427, -0.06309279727001485],
            'AF4': [0.02849366665483203, -0.07313343024976739, -0.06309279727001485],
            'Arm_R_Body': [0.05551410322883508, -0.027015664734477904, -0.03079282724493873],
            'ArmL_Body': [-0.05551410425847653, -0.027015660031019486, -0.030792982403052636],
            'AuxCMS': [-0.06634027689047482, -0.0030793339151980194, 0.012765326982607012],
            'AuxDRL': [0.06634027689047482, -0.0030793339151980194, 0.01276943341869375],
            'CMS': [-0.06563927965347782, -0.006030545767433053, -0.029458098261013078],
            'DRL': [0.06563927965347782, -0.006030545767433053, -0.029458098261013078],
            'F3': [-0.02363707810785255, -0.05494754518077494, -0.0826450100923021],
            'F4': [0.0236370781035902, -0.05494754518077494, -0.0826450100923021],
            'F7': [-0.04149985112566543, -0.0717646495774555, -0.019034005747378804],
            'F8': [0.04149985112566543, -0.0717646495774555, -0.019034005747378804],
            'FC5': [-0.05378159934215332, -0.04925838264772732, -0.050193148788608866],
            'FC6': [0.05378159934215332, -0.04925838264772732, -0.05019314879279459],
            'Headset_Back_Body': [0.0041755319549092755, 0.07179735000299484, -0.013438814943172376],
            'O1': [-0.023960385178350026, 0.08606975499185385, -0.014604999520661032],
            'O2': [0.023960385178350026, 0.08606975499185385, -0.014604999520661032],
            'P7': [-0.05509887202880335, 0.036132210952347504, -0.009576722499429188],
            'P8': [0.05509887202880335, 0.036132210952347504, -0.009576722499429188],
            'T7': [-0.06534174108801341, -0.02824034937506554, -0.013662302267152341],
            'T8': [0.06534174108801341, -0.02824034937506554, -0.013662302267152341],
        }
        
        # Create electrode spheres
        self.electrodes = {}
        for name in electrode_names:
            try:
                # Try to get block by name
                self.electrodes[name] = self.headset.get_block_by_name(name)
                print(f"Found electrode: {name}")
            except Exception as e:
                print(f"Warning: Could not find electrode '{name}' in the model: {e}")
                # If we can't find the electrode, we'll create a placeholder
                # This is just so the script doesn't crash if the model doesn't have all electrodes
                sphere = pv.Sphere(radius=0.01, center=(0, 0, 0))
                sphere.name = name
                self.electrodes[name] = sphere
                            
                if name in self.electrode_name_to_position_center:
                    # Create a sphere at the electrode position
                    position = self.electrode_name_to_position_center[name]
                    sphere = pv.Sphere(radius=0.01, center=position)
                    sphere.name = name
                    # Add the sphere to the plotter with a name
                    self.electrodes[name] = sphere
                    print(f"Created electrode sphere: {name} at position {position}")
                else:
                    print(f"Warning: No position defined for electrode '{name}'")
        
        # Add model to the scene
        self.plotter.add_mesh(self.headset, color='lightgray')
        
        # Add electrodes with initial color (yellow)
        for name, electrode in self.electrodes.items():
            self.plotter.add_mesh(electrode, color='yellow', name=name)
        
        # Add a title
        self.plotter.add_text("EEG Electrode Quality Visualization\nRed = Poor Quality, Green = Good Quality", 
                              position="upper_left", font_size=12, color='white')
        
        # Create interactive widgets
        self.create_widgets()
        
    def create_widgets(self):
        """Create interactive widgets for the visualization"""
        # Create sliders for each electrode
        self.sliders = {}
        self.slider_layout = widgets.Layout(width='300px')
        
        slider_widgets = []
        for name in electrode_names:
            slider = widgets.FloatSlider(
                value=0.5,
                min=0,
                max=1,
                step=0.01,
                description=f'{name}:',
                disabled=False,
                continuous_update=True,
                orientation='horizontal',
                readout=True,
                readout_format='.2f',
                layout=self.slider_layout
            )
            self.sliders[name] = slider
            slider_widgets.append(slider)
        
        # Create buttons for control
        self.random_button = widgets.Button(
            description='Randomize Values',
            button_style='info',
            tooltip='Generate random quality values'
        )
        self.random_button.on_click(self.on_random_button_click)
        
        self.start_stop_button = widgets.Button(
            description='Start Animation',
            button_style='success',
            tooltip='Start/Stop automatic updates'
        )
        self.start_stop_button.on_click(self.on_start_stop_button_click)
        
        self.reset_button = widgets.Button(
            description='Reset Values',
            button_style='warning',
            tooltip='Reset all values to 0.5'
        )
        self.reset_button.on_click(self.on_reset_button_click)
        
        # Create update interval slider
        self.interval_slider = widgets.FloatSlider(
            value=self.update_interval,
            min=0.1,
            max=2.0,
            step=0.1,
            description='Update Interval (s):',
            disabled=False,
            continuous_update=True,
            orientation='horizontal',
            readout=True,
            readout_format='.1f',
            layout=self.slider_layout
        )
        self.interval_slider.observe(self.on_interval_change, names='value')
        
        # Create output widget for status messages
        self.output = widgets.Output()
        
        # Arrange widgets
        self.button_box = widgets.HBox([self.random_button, self.start_stop_button, self.reset_button])
        self.slider_box = widgets.VBox(slider_widgets)
        self.control_box = widgets.VBox([
            widgets.HTML("<h3>Electrode Quality Controls</h3>"),
            self.button_box,
            self.interval_slider,
            widgets.HTML("<h4>Individual Electrode Quality</h4>"),
            self.slider_box,
            self.output
        ])
        
        # Connect sliders to update function
        for name, slider in self.sliders.items():
            slider.observe(self.on_slider_change, names='value')
        
    def on_slider_change(self, change):
        """Handle slider value changes"""
        # Get all current values
        quality_values = [self.sliders[name].value for name in electrode_names]
        # Update visualization
        self.update_visualization(quality_values)
        
    def on_random_button_click(self, b):
        """Handle random button click"""
        quality_values = self.generate_random_quality()
        # Update sliders
        for name, value in zip(electrode_names, quality_values):
            self.sliders[name].value = value
        
    def on_start_stop_button_click(self, b):
        """Handle start/stop button click"""
        if self.running:
            self.running = False
            self.start_stop_button.description = 'Start Animation'
            self.start_stop_button.button_style = 'success'
            with self.output:
                clear_output()
                print("Animation stopped")
        else:
            self.running = True
            self.start_stop_button.description = 'Stop Animation'
            self.start_stop_button.button_style = 'danger'
            with self.output:
                clear_output()
                print("Animation started")
            # Start animation in a separate thread
            import threading
            self.animation_thread = threading.Thread(target=self.animate)
            self.animation_thread.daemon = True
            self.animation_thread.start()
            
    def on_reset_button_click(self, b):
        """Handle reset button click"""
        # Reset all sliders to 0.5
        for name in electrode_names:
            self.sliders[name].value = 0.5
        with self.output:
            clear_output()
            print("Values reset to 0.5")
            
    def on_interval_change(self, change):
        """Handle interval slider change"""
        self.update_interval = change['new']
        
    def animate(self):
        """Run animation loop"""
        while self.running:
            quality_values = self.generate_random_quality()
            # Update sliders (which will trigger visualization update)
            for name, value in zip(electrode_names, quality_values):
                self.sliders[name].value = value
            time.sleep(self.update_interval)
    
    def update_visualization(self, quality_values):
        """
        Update the visualization with new quality values
        This creates a new plot each time instead of updating in-place
        """
        # Create a new plotter
        plotter = pv.Plotter(notebook=True)
        
        # Add the headset
        plotter.add_mesh(self.headset, color='lightgray')
        
        # Add electrodes with updated colors
        for i, name in enumerate(electrode_names):
            if i < len(quality_values) and name in self.electrodes:
                quality = quality_values[i]
                color = [1-quality, quality, 0]  # R,G,B
                plotter.add_mesh(self.electrodes[name], color=color)
        
        # Add title
        plotter.add_text("EEG Electrode Quality Visualization\nRed = Poor Quality, Green = Good Quality", 
                         position="upper_left", font_size=12, color='white')
        
        # Display the updated plot
        with self.output:
            clear_output(wait=True)
            display(plotter.show(jupyter_backend='static'))
            quality_str = ", ".join([f"{name}: {quality:.2f}" for name, quality in zip(electrode_names, quality_values)])
            print(f"Quality values: {quality_str}")
    
    def generate_random_quality(self):
        """Generate random quality values for testing"""
        return [random.uniform(0, 1) for _ in range(len(electrode_names))]
    
    def show(self):
        """Display the visualization and widgets in the notebook"""
        # Initial visualization
        # plot_widget = self.plotter.show(jupyter_backend='static')
        plot_widget = self.plotter.show(jupyter_backend='trame', return_viewer=True)

        # Create dashboard
        dashboard = widgets.VBox([
            plot_widget,
            self.control_box,
            self.output
        ])
        display(dashboard)
        
        # Show initial quality values
        with self.output:
            print("Ready. Adjust sliders or click buttons to update visualization.")


# Create and display the visualizer
visualizer = EEGVisualizer(model_path=r'C:\Users\pho\repos\EmotivEpoc\CyKit\EXTERNAL\meshes\CompleteEmotivEpocEEG.glb', update_interval=0.5)
# visualizer = EEGVisualizer(model_path=r'C:\Users\pho\repos\EmotivEpoc\CyKit\EXTERNAL\meshes\CompleteEmotivEpocEEG1.obj', update_interval=0.5)
visualizer.show()


In [ ]:
visualizer.headset

In [ ]:
visualizer.headset.array_names

In [ ]:
# Print the structure of the loaded model
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any




def extract_headset_mesh_electrode_data(headset_mesh: pv.MultiBlock, debug_print=False):
    electrode_poly_output_dict: Dict[str, pv.PolyData] = {}

    if debug_print:
        print(visualizer.headset)

    electrode_multiblock: pv.MultiBlock = headset_mesh.get_block(0)
    # If it's a MultiBlock, try listing the blocks
    if hasattr(electrode_multiblock, 'n_blocks'):
        # print(f"Number of blocks: {electrode_multiblock.n_blocks}")
        for i in range(electrode_multiblock.n_blocks):
            an_electrode_multiblock: pv.MultiBlock = electrode_multiblock.get_block(i)
            if debug_print:
                print(f"an_electrode_multiblock[{i}]: {type(an_electrode_multiblock)}")
            is_base_multiblock: bool = hasattr(an_electrode_multiblock, 'n_blocks') and (an_electrode_multiblock.n_blocks == 1)
            if is_base_multiblock:
                ## extract root
                # an_electrode_multiblock = an_electrode_multiblock.get_block(0) # PolyData
                an_electrode_polydata: pv.PolyData = an_electrode_multiblock.get_block(0).get_block(0) # PolyData
                electrode_poly_output_dict[i] = deepcopy(an_electrode_polydata)
                if debug_print:
                    print(f'\tFOUND BASE ELECTRODE POLYDATA:')
                    print(f'\t{an_electrode_polydata}')
            else:
                print(f'\tERR: FAILED TO FIND BASE ELECTRODE BLOCK:')
                print(f"\t\tNumber of blocks: {electrode_multiblock.n_blocks}")
                print(f'\t{an_electrode_multiblock}')
            

    ## OUTPUTS: electrode_poly_output_dict
    return electrode_poly_output_dict

electrode_poly_output_dict = extract_headset_mesh_electrode_data(headset_mesh=visualizer.headset)


In [ ]:
an_electrode_polydata.GetNamedFieldInformation()

In [ ]:
## INPUTS: electrode_poly_output_dict

self.electrode_names_dict = {
    0:"AF3",
    1:"AF4",
    2:"Arm_R_Body",
    3:"ArmL_Body",
    4:"AuxCMS",
    5:"AuxDRL",
    6:"CMS",
    7:"DRL",
    8:"F3",
    9:"F4",
    10:"F7",
    11:"F8",
    12:"FC5",
    13:"FC6",
    14:"Headset_Back_Body",
    15:"O1",
    16:"O2",
    17:"P7",
    18:"P8",
    19:"T7",
    20:"T8",	
}

assert len(self.electrode_names_dict) == len(electrode_poly_output_dict), f"len(electrode_names_dict): {len(self.electrode_names_dict)} != len(electrode_poly_output_dict): {len(electrode_poly_output_dict)}"

{self.electrode_names_dict[k]:v.center_of_mass().tolist() for k, v in electrode_poly_output_dict.items()}





# an_electrode_polydata.center_of_mass

In [ ]:
print(list(self.electrode_names_dict.values()))

# ['AF3', 'AF4', 'Arm_R_Body', 'ArmL_Body', 'AuxCMS', 'AuxDRL', 'CMS', 'DRL', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'Headset_Back_Body', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8']
['AF3', 'AF4', 'AuxCMS', 'AuxDRL', 'CMS', 'DRL', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8']

In [ ]:


an_electrode_multiblock = electrode_multiblock.get_block(0).get_block(0)

# electrode_multiblock
an_electrode_multiblock

In [ ]:

# If it's a MultiBlock, try listing the blocks
if hasattr(visualizer.headset, 'n_blocks'):
    print(f"Number of blocks: {visualizer.headset.n_blocks}")
    for i in range(visualizer.headset.n_blocks):
        block = visualizer.headset.get_block(i)
        print(f"Block {i}: {type(block)}")
        print(f'\tblock: {block}')